In [1]:
import pandas as pd
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

from scipy.spatial import ConvexHull

pd.set_option('display.max_columns', None)

In [2]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.python.worker.faulthandler.enabled", "true") # se algum worker crashar de novo, imprime o traceback nativo real
    .master("local[*]")
    .appName("sanity_features_tracking")
    .getOrCreate()
)

In [3]:
def plot_threat_event(df_threat, show_player_names=False, show_player_positions=False):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal[_zona]
    - defenders_between_ball_goal[_zona]
    - total_players[_zona]
    - atk_def_advantage[_zona]

    Espera um DataFrame Spark contendo exatamente um evento.

    Sempre desenha: linha da bola, linha do meio-campo (x=0) e as
    2 linhas que dividem o campo em 3 terços.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    show_labels = show_player_names or show_player_positions

    def player_label(p):
        parts = []
        if show_player_names:
            parts.append(p["player"]["name"])
        if show_player_positions:
            parts.append(f'({p["position"]})')
        return " ".join(parts) if parts else None

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_labels else "markers",
        text=[player_label(p) for p in attackers] if show_labels else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_labels else "markers",
        text=[player_label(p) for p in defenders] if show_labels else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)

    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Linhas divisórias fixas: meio-campo + 2 terços
    # ============================

    field_length = right_x - left_x

    zone_boundaries = [
        0.0,                                # meio-campo (half)
        #left_x + field_length / 3,          # início do terço 2
        #left_x + 2 * field_length / 3,      # início do terço 3
    ]

    for x_boundary in zone_boundaries:
        fig.add_vline(
            x=x_boundary,
            line_dash="dot",
            line_width=2,
            line_color="gray"
        )

    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )

    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)

    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Evento: {row['eventTypeDescription']} | "
        f"Posse: {row['eventTeamName']} | "
        f"P. Invertida: {row['flipped_homeTeam']} | "
        f"Ameaça: {row['threat_score']:.3f} | "
        f"Impacto: {row['threat_score_impact']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
    )

    fig.show()

In [4]:
def base_defenders_figure(def_x, def_y, def_labels):
    """
    Cria a figura Plotly base reaproveitada pelas Features 1-4: só os
    defensores de linha (sem goleiro) plotados como markers rotulados pela
    posição.

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.

    Retorna
    -------
    plotly.graph_objects.Figure
    """
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=def_x, y=def_y, mode='markers+text',
        text=def_labels, textposition='top center',
        marker=dict(size=12, color='blue'), name='Defensores (sem GK)'
    ))
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.update_layout(height=550, width=700)
    return fig


def plot_surface_area(def_x, def_y, def_labels, surface_area):
    """
    Plota o casco convexo (Feature 1) sobre os defensores de linha (sem GK),
    preenchendo a área calculada.

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    surface_area : float
        Valor de surface_area já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    pts = np.array(sorted(set(zip(def_x.tolist(), def_y.tolist()))))
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        hull_pts = np.vstack([pts[hull.vertices], pts[hull.vertices[0]]])
        fig.add_trace(go.Scatter(
            x=hull_pts[:, 0], y=hull_pts[:, 1], mode='lines', fill='toself',
            fillcolor='rgba(0,0,255,0.1)', line=dict(color='blue', dash='dot'),
            name='Casco convexo'
        ))

    fig.update_layout(title=f"surface_area = {surface_area} m²")
    fig.show()


def plot_stretch_index(def_x, def_y, def_labels, stretch_index):
    """
    Plota o centroide dos defensores de linha (sem GK) e uma linha de cada
    defensor até ele (Feature 2).

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    stretch_index : float
        Valor de stretch_index já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    cx, cy = def_x.mean(), def_y.mean()
    fig.add_trace(go.Scatter(x=[cx], y=[cy], mode='markers', marker=dict(size=12, symbol='x', color='purple'), name='Centroide'))
    for i, (x, y) in enumerate(zip(def_x, def_y)):
        fig.add_trace(go.Scatter(
            x=[cx, x], y=[cy, y], mode='lines', line=dict(color='purple', width=1),
            name='Distância até o centroide' if i == 0 else None, showlegend=(i == 0)
        ))

    fig.update_layout(title=f"stretch_index = {stretch_index} m (média das linhas roxas)")
    fig.show()


def plot_team_length(def_x, def_y, def_labels, team_length):
    """
    Plota linhas verticais tracejadas no defensor mais atrás e no mais à
    frente (eixo x) e um segmento horizontal ligando as duas (Feature 3).

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    team_length : float
        Valor de team_length já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    i_min, i_max = int(def_x.argmin()), int(def_x.argmax())
    fig.add_vline(x=def_x[i_min], line_dash='dash', line_color='green', annotation_text='Defensor mais atrás')
    fig.add_vline(x=def_x[i_max], line_dash='dash', line_color='green', annotation_text='Defensor mais à frente')
    fig.add_trace(go.Scatter(
        x=[def_x[i_min], def_x[i_max]], y=[def_y.min() - 3, def_y.min() - 3],
        mode='lines+markers', line=dict(color='green', width=3), marker=dict(size=14, color='green'),
        name='team_length'
    ))

    fig.update_layout(title=f"team_length = {team_length} m")
    fig.show()


def plot_height_goal_player(def_x, def_y, def_labels, goal_x, height_goal_player):
    """
    Plota a linha do gol do time defendendo (x = goal_x) e um segmento até o
    defensor de linha mais próximo dela (Feature 4).

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    goal_x : float
        Posição x do gol do time defendendo (stadiumLength / 2, já que o
        ataque está sempre normalizado pra direita).
    height_goal_player : float
        Valor de height_goal_player já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    i_closest = int(def_x.argmax())  # ataque normalizado pra direita -> gol em +goal_x, defensor mais próximo = maior x
    fig.add_vline(x=goal_x, line_dash='dash', line_color='black', annotation_text='Linha do gol')
    fig.add_trace(go.Scatter(
        x=[def_x[i_closest], goal_x], y=[def_y[i_closest], def_y[i_closest]],
        mode='lines+markers', line=dict(color='red', width=3), marker=dict(size=14, color='red'),
        name='height_goal_player'
    ))

    fig.update_layout(title=f"height_goal_player = {height_goal_player} m")
    fig.show()


def plot_numeric_superiority(all_attackers, all_defenders, ball, radius, value, radius_color='orange'):
    """
    Plota a bola, um círculo de raio `radius` ao redor dela e todos os
    atacantes/defensores (com goleiro) — usada pras Features 6 e 7
    (numeric_superiority_10m/20m).

    Parâmetros
    ----------
    all_attackers, all_defenders : list[Row]
        Jogadores completos (com GK) do time atacante/defendendo do evento,
        cada um com pelo menos os campos 'x' e 'y'.
    ball : Row
        Posição da bola do evento (campos 'x'/'y').
    radius : float
        Raio (m) ao redor da bola usado na contagem de superioridade numérica.
    value : float
        Valor de numeric_superiority_{radius}m já calculado pra esse evento
        (exibido no título).
    radius_color : str
        Cor da linha do círculo do raio (default 'orange').
    """
    fig = go.Figure()

    theta = np.linspace(0, 2 * np.pi, 100)
    fig.add_trace(go.Scatter(
        x=ball['x'] + radius * np.cos(theta), y=ball['y'] + radius * np.sin(theta),
        mode='lines', line=dict(color=radius_color, dash='dot'), name=f'Raio {radius}m'
    ))
    fig.add_trace(go.Scatter(
        x=[p['x'] for p in all_attackers], y=[p['y'] for p in all_attackers],
        mode='markers', marker=dict(size=10, color='red'), name='Atacantes (com GK)'
    ))
    fig.add_trace(go.Scatter(
        x=[p['x'] for p in all_defenders], y=[p['y'] for p in all_defenders],
        mode='markers', marker=dict(size=10, color='blue'), name='Defensores (com GK)'
    ))
    fig.add_trace(go.Scatter(x=[ball['x']], y=[ball['y']], mode='markers', marker=dict(size=14, color='black'), name='Bola'))

    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.update_layout(height=550, width=700, title=f"numeric_superiority_{radius}m = {value}")
    fig.show()

In [5]:
def plot_histogram(df, column, nbins=None, title=None, color="#2E5EAA"):
    pdf = df.select(column).toPandas()

    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=pdf[column],
            nbinsx=nbins,
            marker=dict(color=color),
            name=column,
        )
    )

    fig.update_layout(
        template="simple_white",
        title=title or f"Distribuição de {column}",
        xaxis_title=column,
        yaxis_title="Frequência",
        height=600,
        width=1000
    )

    fig.show()
    
def plot_boxplot(df, column, title=None, color="#2E5EAA"):
    pdf = df.select(column).toPandas()

    fig = go.Figure()
    fig.add_trace(
        go.Box(
            y=pdf[column],
            marker=dict(color=color),
            name=column,
            boxpoints="outliers",
        )
    )

    fig.update_layout(
        template="simple_white",
        title=title or f"Boxplot de {column}",
        yaxis_title=column,
        height=600,
        width=1000
    )

    fig.show()
    
def plot_line(df, column, x=None, title=None, color="#2E5EAA"):
    cols = [x, column] if x is not None else [column]
    pdf = df.select(*cols).toPandas()

    x_values = pdf[x] if x is not None else pdf.index
    median = float(pdf[column].median())

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=pdf[column],
            mode="lines",
            line=dict(color=color, width=2),
            name=column,
        )
    )

    fig.add_shape(
        type="line",
        xref="paper",
        x0=0,
        x1=1,
        yref="y",
        y0=median,
        y1=median,
        line=dict(color="red", dash="dash", width=2),
    )

    fig.update_layout(
        template="simple_white",
        title=title or f"{column} ao longo do tempo",
        xaxis_title=x or "Índice",
        yaxis_title=column,
        height=600,
        width=1500,
    )

    fig.show()

In [6]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [7]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
features_dataset_path = str(data_folder_path / "features_dataset")

df_threat = spark.read.parquet(threat_dataset_path)
df_features = spark.read.csv(features_dataset_path, header=True, inferSchema=True)

In [8]:
threat_cols = [
    'gameId',
    'competitionId',
    'season',
    'date',
    'eventId',
    'period',
    'startGameClock',
    'startFormattedGameClock',
    'homeTeamName',
    'opponentTeamName',
    'homeTeam',
    #'flipped_homeTeam',
    #'eventType',
    'eventTypeDescription',
    #'eventSubTypeDescription',
    'eventOutcomeDescription',
    #'eventPlayerName',
    #'eventPlayerPositionType',
    #'eventPlayerPositionGroup',
    'eventTeamName',
    'possession_id',
    'attackingPlayersNorm',
    'defendingPlayersNorm',
    'threat_score',
    'threat_score_impact'
]

In [9]:
#df_threat.columns

In [10]:
df_threat_features = (
    df_threat.select(*threat_cols)
    .join(
        df_features,
        on=["competitionId", "season", "gameId", "eventId"],
        how='inner'
    )
)

## Diagnóstico de inconsistências no tracking

Investiga casos onde `surface_area`/`stretch_index` caem pra valores muito
baixos: hipótese é que o provider de tracking, quando perde vários jogadores,
passa a estimar todos na mesma posição (`visibility='ESTIMATED'`,
`confidence='LOW'`), o que colapsa o convex hull / dispersão em torno do
centroide após a dedup de coordenadas.

In [11]:
def mode_of_array(arr_col):
    """
    Retorna o valor mais frequente (moda) de um array de strings, via
    higher-order functions do Spark (sem UDF). eqNullSafe trata NULL como um
    valor comparável (jogador sem visibility/confidence tracked).
    """
    distinct_vals = F.array_distinct(arr_col)
    counted = F.transform(
        distinct_vals,
        lambda v: F.struct(
            (-F.size(F.filter(arr_col, lambda y: y.eqNullSafe(v)))).alias("neg_cnt"),
            v.alias("val"),
        )
    )
    return F.element_at(F.array_sort(counted), 1)["val"]


def pct_matching(arr_col, value):
    """Fração de elementos do array (array de structs de jogador) cujo campo já extraído == value."""
    return F.size(F.filter(arr_col, lambda y: y.eqNullSafe(value))) / F.size(arr_col)


def distinct_coords_count(players_col):
    """Quantidade de pares (x, y) distintos entre os jogadores do array — mede o colapso de coordenadas."""
    coords = F.transform(players_col, lambda p: F.struct(p["x"].alias("x"), p["y"].alias("y")))
    return F.size(F.array_distinct(coords))


def max_shared_coord_count(players_col):
    """
    Maior quantidade de jogadores compartilhando exatamente a mesma
    coordenada (x, y) — mais informativo que a contagem de distintos:
    2 jogadores dividindo o mesmo ponto é plausível (disputa de bola),
    mas 3+ jogadores no EXATO mesmo ponto (mesmo com o ruído de medição
    do tracking) não é fisicamente plausível em campo.
    """
    coords = F.transform(players_col, lambda p: F.struct(p["x"].alias("x"), p["y"].alias("y")))
    distinct_coords = F.array_distinct(coords)
    group_sizes = F.transform(distinct_coords, lambda c: F.size(F.filter(coords, lambda y: y.eqNullSafe(c))))
    return F.array_max(group_sizes)


attacking_visibility = F.transform("attackingPlayersNorm", lambda p: p["visibility"])
defending_visibility = F.transform("defendingPlayersNorm", lambda p: p["visibility"])
attacking_confidence = F.transform("attackingPlayersNorm", lambda p: p["confidence"])
defending_confidence = F.transform("defendingPlayersNorm", lambda p: p["confidence"])
defending_outfield = F.filter("defendingPlayersNorm", lambda p: p["position"]["type"] != "GK")

df_tracking_diagnostics = df_threat_features.withColumns({
    # 1.1 / 1.2 — quantidade de jogadores
    "attacking_players": F.size("attackingPlayersNorm"),
    "defending_players": F.size("defendingPlayersNorm"),
    "defending_outfield_players": F.size(defending_outfield),

    # 1.3 a 1.6 — moda de visibility/confidence
    "attackers_avg_visibility": mode_of_array(attacking_visibility),
    "defenders_avg_visibility": mode_of_array(defending_visibility),
    "attackers_avg_confidence": mode_of_array(attacking_confidence),
    "defenders_avg_confidence": mode_of_array(defending_confidence),

    # extra — % de jogadores em baixa confiança/estimados (mais sensível que a moda)
    "attackers_pct_low_confidence": pct_matching(attacking_confidence, "LOW"),
    "defenders_pct_low_confidence": pct_matching(defending_confidence, "LOW"),
    "attackers_pct_estimated": pct_matching(attacking_visibility, "ESTIMATED"),
    "defenders_pct_estimated": pct_matching(defending_visibility, "ESTIMATED"),

    # extra — % de jogadores VISIBLE (complemento de pct_estimated): serve
    # pra ver se um pct médio de VISIBLE maior está associado aos casos de
    # tracking consistente na análise seguinte
    "attackers_pct_visible": pct_matching(attacking_visibility, "VISIBLE"),
    "defenders_pct_visible": pct_matching(defending_visibility, "VISIBLE"),

    # extra — coordenadas (x, y) distintas entre os defensores de linha:
    # é exatamente o que compute_surface_area/compute_stretch_index usam
    # após a dedup, então mede diretamente o colapso do casco/dispersão
    "defending_outfield_distinct_coords": distinct_coords_count(defending_outfield),
    "defending_outfield_max_shared_coord_count": max_shared_coord_count(defending_outfield),
})

df_tracking_diagnostics = df_tracking_diagnostics.withColumn(
    "defending_outfield_unique_ratio",
    F.when(
        F.col("defending_outfield_players") > 0,
        F.round(F.col("defending_outfield_distinct_coords") / F.col("defending_outfield_players"), 3)
    )
)

df_tracking_diagnostics.select(
    "eventId", "defending_outfield_players", "defending_outfield_distinct_coords",
    "defending_outfield_unique_ratio", "defending_outfield_max_shared_coord_count",
    "defenders_avg_visibility", "defenders_avg_confidence",
    "defenders_pct_low_confidence", "defenders_pct_estimated", "defenders_pct_visible",
    "surface_area", "stretch_index",
).show(10, truncate=False)

+--------------------------------+--------------------------+----------------------------------+-------------------------------+-----------------------------------------+------------------------+------------------------+----------------------------+-----------------------+---------------------+------------+-------------+
|eventId                         |defending_outfield_players|defending_outfield_distinct_coords|defending_outfield_unique_ratio|defending_outfield_max_shared_coord_count|defenders_avg_visibility|defenders_avg_confidence|defenders_pct_low_confidence|defenders_pct_estimated|defenders_pct_visible|surface_area|stretch_index|
+--------------------------------+--------------------------+----------------------------------+-------------------------------+-----------------------------------------+------------------------+------------------------+----------------------------+-----------------------+---------------------+------------+-------------+
|04ba415ceee9f81967ff8d544b42d9

### Deixando o limiar ser sugerido pelos dados

Em vez de fixar um limiar arbitrário na razão de coordenadas únicas, olha a
distribuição de `defending_outfield_max_shared_coord_count` (maior grupo de
defensores de linha compartilhando exatamente a mesma coordenada) em toda a
base: no futebol é raro mas plausível **2** jogadores dividirem o mesmo
espaço numa disputa de bola — já **3 ou mais** jogadores no EXATO mesmo ponto
não tem explicação tática, mesmo com o ruído normal de medição do tracking.
A contagem por valor abaixo confirma (ou não) que existe uma queda clara a
partir de 3, o que valida o limiar.

In [12]:
(
    df_tracking_diagnostics
    .groupBy("defending_outfield_max_shared_coord_count")
    .count()
    .withColumn("pct_do_total", F.round(F.col("count") / df_tracking_diagnostics.count() * 100, 3))
    .orderBy("defending_outfield_max_shared_coord_count")
    .show(50)
)

+-----------------------------------------+------+------------+
|defending_outfield_max_shared_coord_count| count|pct_do_total|
+-----------------------------------------+------+------------+
|                                        1|430937|       96.15|
|                                        2|  2187|       0.488|
|                                        3|  2077|       0.463|
|                                        4|  3717|       0.829|
|                                        5|  3880|       0.866|
|                                        6|  2630|       0.587|
|                                        7|  1373|       0.306|
|                                        8|  1101|       0.246|
|                                        9|   253|       0.056|
|                                       10|    38|       0.008|
+-----------------------------------------+------+------------+



### Identificando os casos inconsistentes

Flag `is_tracking_inconsistent`: `defending_outfield_max_shared_coord_count >= 2`
— qualquer caso com pelo menos 2 defensores de linha tracked exatamente na
mesma coordenada já é considerado inconsistente (mesmo uma disputa de bola
legítima não deveria produzir coordenadas idênticas — só próximas). Análises
baseadas em distância entre jogadores (vizinho mais próximo) foram
descartadas — só posições exatamente iguais contam. Comparado contra
`defenders_pct_low_confidence`/`defenders_pct_estimated` (baixa confiança
concentrada nos casos flagados confirma que é degradação de tracking) e
contra a moda de `eventTypeDescription` (pra ver qual tipo de evento
concentra mais inconsistências).

In [ ]:
MIN_SHARED_COORD_THRESHOLD = 2  # ver distribuição empírica na célula anterior

df_tracking_diagnostics = df_tracking_diagnostics.withColumn(
    "is_tracking_inconsistent",
    F.col("defending_outfield_max_shared_coord_count") >= MIN_SHARED_COORD_THRESHOLD
)

qtd_inconsistentes = df_tracking_diagnostics.filter(F.col("is_tracking_inconsistent")).count()
qtd_total = df_tracking_diagnostics.count()
print(f"Eventos com tracking inconsistente: {qtd_inconsistentes} / {qtd_total} ({qtd_inconsistentes / qtd_total:.2%})")

qtd_jogos_inconsistentes = df_tracking_diagnostics.filter(F.col("is_tracking_inconsistent")).select("gameId").distinct().count()
qtd_jogos_total = df_tracking_diagnostics.select("gameId").distinct().count()
print(f"Jogos com pelo menos 1 evento inconsistente: {qtd_jogos_inconsistentes} / {qtd_jogos_total} ({qtd_jogos_inconsistentes / qtd_jogos_total:.2%})")

# Cruza o flag com surface_area/stretch_index e com os indicadores de confidence/visibility,
# pra confirmar se o colapso das features coincide com tracking degradado
(
    df_tracking_diagnostics
    .groupBy("is_tracking_inconsistent")
    .agg(
        F.count("*").alias("qtd_eventos"),
        
        F.round(F.mean("defenders_pct_low_confidence"), 3).alias("avg_pct_low_confidence"),
        F.round(F.mean("defenders_pct_estimated"), 3).alias("avg_pct_estimated"),
        F.round(F.mean("defenders_pct_visible"), 3).alias("avg_pct_visible"),
        
        F.countDistinct("gameId").alias("qtd_jogos"),
        
        # Forma/dispersão do time defendendo
        F.round(F.mean("surface_area"), 2).alias("avg_surface_area"),
        
        F.round(F.mean("stretch_index"), 2).alias("avg_stretch_index"),
        
        F.round(F.mean("team_length"), 2).alias("avg_team_length"),
        F.round(F.mean("team_width"), 2).alias("avg_team_width"),
        F.round(F.mean("defense_width"), 2).alias("avg_defense_width"),
        
        # Altura da defesa em relação ao próprio gol
        F.round(F.mean("height_goal_player"), 2).alias("avg_height_goal_player"),
        F.round(F.mean("height_goal_team_centroide"), 2).alias("avg_height_goal_team_centroide"),
        F.round(F.mean("height_goal_def_centroide"), 2).alias("avg_height_goal_def_centroide"),
        
        # Distância longitudinal entre linhas (centroides por groupType)
        F.round(F.mean("def_mid_dist"), 2).alias("avg_def_mid_dist"),
        F.round(F.mean("def_atk_dist"), 2).alias("avg_def_atk_dist"),
        F.round(F.mean("atk_mid_dist"), 2).alias("avg_atk_mid_dist"),
        
        # Superioridade numérica ao redor da bola
        F.round(F.mean("numeric_superiority_10m"), 2).alias("avg_numeric_superiority_10m"),        
        F.round(F.mean("numeric_superiority_20m"), 2).alias("avg_numeric_superiority_20m"),
        
        F.round(F.mean("threat_score"), 4).alias("avg_threat_score"),
        F.round(F.mean("threat_score_impact"), 4).alias("avg_threat_score_impact"),
    )
    .show(truncate=False)
)

Eventos com tracking inconsistente: 17256 / 448193 (3.85%)
Jogos com pelo menos 1 evento inconsistente: 284 / 361 (78.67%)
+------------------------+-----------+----------------------+-----------------+---------------+---------+----------------+-----------------+---------------+--------------+-----------------+----------------------+------------------------------+-----------------------------+----------------+----------------+----------------+---------------------------+---------------------------+----------------+-----------------------+
|is_tracking_inconsistent|qtd_eventos|avg_pct_low_confidence|avg_pct_estimated|avg_pct_visible|qtd_jogos|avg_surface_area|avg_stretch_index|avg_team_length|avg_team_width|avg_defense_width|avg_height_goal_player|avg_height_goal_team_centroide|avg_height_goal_def_centroide|avg_def_mid_dist|avg_def_atk_dist|avg_atk_mid_dist|avg_numeric_superiority_10m|avg_numeric_superiority_20m|avg_threat_score|avg_threat_score_impact|
+------------------------+-------

Observa-se que 96.15% dos eventos não possuem conflito de coordenadas (x,y). Isso é uma volumetria saudável, permitindo remover os 3.85% incosistentes sem impactar na população dos dados. 

Além disso:
- Pode-se ver que é nitido o impacto que as features são diferentes e sofrem pelo fato de ocorrer incosistência nos dados de tracking. 
- Também é visto uma diferença nitida no percentual média de registros estimados e com confiança baixa. Além disso, são dados com um percentual baixissimo de tracking visivel (VISIBLE).
- Por fim, a ameaça também não sofre muito com as incosistências de tracking, sendo menos sensivel. Até faz sentido, visto que a componente com maior correlação é a progression_distance.

Logo, podemos seguir com a premissa de que pelo menos 2 jogadores compartilhando coordenadas exatas é fisicamente impossivel e podem ser removidos.